In [1]:
import os
import numpy as np
import h5py
from scipy import signal
from scipy.stats import pearsonr
import seaborn as sns
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import string

# Define path to data:
script_dir = os.path.dirname(os.path.abspath("Figure 1 Data Summary.ipynb"))
parent_dir = os.path.dirname(script_dir)
data_path = (parent_dir + '/dfs/')

In [6]:
# Load device, cshq, and sleep diary data file:
df_dreem = pd.read_csv(data_path+'df_dreem_git.csv')
df_embrace = pd.read_csv(data_path+'df_embrace_git.csv')
df_withings = pd.read_csv(data_path+'df_withings_git.csv')
df_yasa = pd.read_csv(data_path+'df_yasa_git.csv')
df_diary = pd.read_csv(data_path+'df_diary_git.csv')

# Load subject demographics:
df_demographics = pd.read_csv(data_path+'SSP_participants.csv')

print(len(df_dreem), len(df_embrace), len(df_withings), len(df_yasa), len(df_diary), len(df_demographics))

2534 2426 3169 2418 2630 200


In [7]:
# Remove timezone (last six values) from SO and FA fields in data and format into datetime field:

def format_datetime_columns(df, columns):
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column].str.slice(stop=-6))
    return df

# Apply function to format datetime columns:
df_dreem = format_datetime_columns(df_dreem, ['SO', 'FA', 'start_rec'])
df_withings = format_datetime_columns(df_withings, ['SO', 'FA'])
df_embrace = format_datetime_columns(df_embrace, ['SO', 'FA'])

# There is no timezone in YASA or diary data, so we can directly convert to datetime:
df_yasa['SO'] = pd.to_datetime(df_yasa['SO'])
df_yasa['FA'] = pd.to_datetime(df_yasa['FA'])
df_yasa['start_rec'] = pd.to_datetime(df_yasa['start_rec'])

df_diary['SO'] = pd.to_datetime(df_diary['SO'])
df_diary['FA'] = pd.to_datetime(df_diary['FA'])

# Caluclate time from midnight for SO and FA fields:

def calculate_from_midnight(df, so_column, fa_column, start_rec):
    df['SO_FromMidNight'] = (df[so_column].dt.hour + df[so_column].dt.minute / 60 + df[so_column].dt.second / 3600)
    df['FA_FromMidNight'] = (df[fa_column].dt.hour + df[fa_column].dt.minute / 60 + df[fa_column].dt.second / 3600)
    
    # Create a boolean variable testing wether SO and FA were on the same date (child went to sleep after midnight)
    condition = df[so_column].dt.date == df[fa_column].dt.date
    # When true add 24 hours to SO
    df.loc[condition, 'SO_FromMidNight'] = df.loc[condition, 'SO_FromMidNight'] + 24
    # Do the same for start recording field if it exists:
    if start_rec != '':
        df['start_rec_FromMidNight'] = (df[start_rec].dt.hour + df[start_rec].dt.minute / 60 + df[start_rec].dt.second / 3600)
        condition = df[start_rec].dt.date == df[fa_column].dt.date
        df.loc[condition, 'start_rec_FromMidNight'] = df.loc[condition, 'start_rec_FromMidNight'] + 24
    # Always add 24 hours to FA
    df['FA_FromMidNight'] = df['FA_FromMidNight'] + 24

    return df


# Apply function to format datetime columns:
df_dreem = calculate_from_midnight(df_dreem, 'SO', 'FA', 'start_rec')
df_yasa = calculate_from_midnight(df_yasa, 'SO', 'FA', 'start_rec')
df_withings = calculate_from_midnight(df_withings, 'SO', 'FA', '')
df_embrace = calculate_from_midnight(df_embrace, 'SO', 'FA', '')
df_diary = calculate_from_midnight(df_diary, 'SO', 'FA', '')


In [8]:
# List of data types
data_types = ['dreem', 'yasa', 'withings', 'embrace', 'diary']
nights_lost = [0, 0, 0, 0, 0]

# Dictionary to store clean dataframes
clean_dataframes = {}

# Loop through each data type
for data_type in data_types:
    # Get the dataframe
    df = globals()[f'df_{data_type}']
    night_count = len(df)
    
    # Clean the data:
    # Keep all nights where SO was after 7pm on day preceding FA and before 7am on day of FA
    df_clean = df[df['SO_FromMidNight'] > 19]
    df_clean = df_clean[df_clean['SO_FromMidNight'] < 31]

    # Keep all nights where TST > 3 hours and TST < 16 hours
    df_clean = df_clean[df_clean['TST'] < 960]
    df_clean = df_clean[df_clean['TST'] > 180]

    # Keep all nights with WASO < 3 hours
    df_clean = df_clean[df_clean['WASO'] < 180]

    # Keep all nights with FA before 2pm on day of FA
    df_clean = df_clean[df_clean['FA_FromMidNight'] < 38]

    nights_lost[data_types.index(data_type)] = night_count - len(df_clean)

    # If data_type == 'dreem' or 'yasa', add an SOL column:
    if data_type == 'dreem' or data_type == 'yasa':
        df_clean['SOL'] = (df_clean['SO_FromMidNight'] - df_clean['start_rec_FromMidNight'])*60
        # Limit SOL to 180 minutes, replace with NaN if greater:
        df_clean.loc[df_clean['SOL'] > 180, 'SOL'] = np.nan

    # Save clean data file
    df_clean.to_csv(data_path + f'df_{data_type}_clean.csv', index=False)
    
    # Store the clean dataframe in the dictionary
    clean_dataframes[f'df_{data_type}_clean'] = df_clean

# Pull out cleaned data from dictionary:
df_dreem_clean = clean_dataframes['df_dreem_clean']
df_yasa_clean = clean_dataframes['df_yasa_clean']
df_embrace_clean = clean_dataframes['df_embrace_clean']
df_withings_clean = clean_dataframes['df_withings_clean']
df_diary_clean = clean_dataframes['df_diary_clean']

# Save nights lost count:
nights_lost_dict = dict(zip(data_types, nights_lost))
nights_lost_df = pd.DataFrame(nights_lost_dict.items(), columns=['Data Type', 'Nights Lost'])
nights_lost_df.to_csv(data_path + 'nights_lost.csv', index=False)

print(len(df_dreem_clean), len(df_yasa_clean),  len(df_embrace_clean), len(df_withings_clean), len(df_diary_clean))

2408 2271 2348 3040 2624
